# Sorting Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

%md
## Que1: Find Total Time Spent by Each Employee

**Difficulty:** Easy

### Problem

An office building records every time an employee badges in and out. Within a single day, an employee may enter and leave more than once.

For every employee and every day they visited the office, calculate the total number of minutes spent inside the office by summing the duration of all visits for that day.

**Schema columns:** `employees.emp_id`, `employees.event_day`, `employees.in_time`, `employees.out_time`

**Output columns:** `day`, `emp_id`, `total_time`

Order the result by `day` in ascending order, then by `emp_id` in ascending order.

### Examples

#### Example 1

**Input:**

**employees:**

| emp_id | event_day | in_time | out_time |
|-------:|------------|--------:|---------:|
| 1 | 2020-11-28 | 4 | 32 |
| 1 | 2020-11-28 | 55 | 200 |
| 1 | 2020-12-03 | 1 | 42 |
| 2 | 2020-11-28 | 3 | 33 |
| 2 | 2020-12-09 | 47 | 74 |

**Output:**

| day | emp_id | total_time |
|------------|-------:|-----------:|
| 2020-11-28 | 1 | 173 |
| 2020-11-28 | 2 | 30 |
| 2020-12-03 | 1 | 41 |
| 2020-12-09 | 2 | 27 |

**Explanation:** On 2020-11-28, employee 1 had two visits lasting 28 and 145 minutes, giving a total of 173 minutes. Employee 2 had a single 30-minute visit. Each employee's daily visits are summed independently.

### Constraints

- `in_time` is always less than `out_time`.
- An employee may have multiple visits on the same day.
- Sum all visit durations for each `(day, emp_id)` pair.
- Return one row per employee per day.
- Order the result by `day`, then `emp_id`.

In [0]:
employees_data=[(1,"2020-11-28",4,32),(1,"2020-11-28",55,200),(1,"2020-12-03",1,42),(2,"2020-11-28",3,33),(2,"2020-12-09",47,74)]
employees_df=spark.createDataFrame(employees_data,["emp_id","event_day","in_time","out_time"])
display(employees_df)

duration_df = (
employees_df
    .withColumn("duration", col("out_time") - col("in_time"))
)

grouped_df = (
duration_df.groupBy("emp_id", "event_day").agg(
    sum("duration").alias("total_timr")
)).orderBy("event_day", "emp_id")

display(grouped_df)


emp_id,event_day,in_time,out_time
1,2020-11-28,4,32
1,2020-11-28,55,200
1,2020-12-03,1,42
2,2020-11-28,3,33
2,2020-12-09,47,74


emp_id,event_day,total_timr
1,2020-11-28,173
2,2020-11-28,30
1,2020-12-03,41
2,2020-12-09,27


%md
## Que2: Trending Tweet Tracker

**Difficulty:** Medium

### Problem

A social platform tracks the number of tweets posted by each user every day.

For every user and tweet date, calculate the rolling average of `tweet_count` using the current row and the previous two available rows for the same user. If fewer than three rows are available, calculate the average using only the available rows.

Round the rolling average to 2 decimal places.

**Schema columns:** `ttt_tweet_input.user_id`, `ttt_tweet_input.tweet_date`, `ttt_tweet_input.tweet_count`

**Output columns:** `user_id`, `tweet_date`, `rolling_avg_3d`

Order the result by `user_id` in ascending order, then by `tweet_date` in ascending order.

### Examples

#### Example 1

**Input:**

**ttt_tweet_input:**

| user_id | tweet_date | tweet_count |
|--------:|---------------------|------------:|
| 111 | 2022-06-01T00:00:00 | 2 |
| 111 | 2022-06-02T00:00:00 | 1 |
| 111 | 2022-06-03T00:00:00 | 3 |
| 111 | 2022-06-04T00:00:00 | 4 |
| 111 | 2022-06-05T00:00:00 | 5 |

**Output:**

| user_id | tweet_date | rolling_avg_3d |
|--------:|------------|---------------:|
| 111 | 2022-06-01 | 2.00 |
| 111 | 2022-06-02 | 1.50 |
| 111 | 2022-06-03 | 2.00 |
| 111 | 2022-06-04 | 2.67 |
| 111 | 2022-06-05 | 4.00 |

**Explanation:** For each row, compute the average tweet count using the current row and the previous two rows for the same user. On 2022-06-04, the values are 1, 3, and 4, resulting in an average of 2.67.

### Constraints

- Calculate the rolling average independently for each user.
- Use the current row and the previous two available rows.
- Round `rolling_avg_3d` to 2 decimal places.
- Order the result by `user_id`, then `tweet_date`.

In [0]:
ttt_tweet_input_data=[(111,"2022-06-01T00:00:00",2),(111,"2022-06-02T00:00:00",1),(111,"2022-06-03T00:00:00",3),(111,"2022-06-04T00:00:00",4),(111,"2022-06-05T00:00:00",5)]

ttt_tweet_input_df=spark.createDataFrame(ttt_tweet_input_data,["user_id","tweet_date","tweet_count"])

window = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(-2, 0)

output_df = (
ttt_tweet_input_df.withColumn("rolling_avg_3", round( avg("tweet_count").over(window), 2))
.orderBy("user_id", "tweet_date")
)

display(output_df)

user_id,tweet_date,tweet_count,rolling_avg_3
111,2022-06-01T00:00:00,2,2.0
111,2022-06-02T00:00:00,1,1.5
111,2022-06-03T00:00:00,3,2.0
111,2022-06-04T00:00:00,4,2.67
111,2022-06-05T00:00:00,5,4.0


%md
## Que3: FAANG Stock Price Analysis

**Difficulty:** Medium

### Problem

You are analyzing historical stock prices for major tech companies.

For each stock ticker, find the month with the highest opening price and the month with the lowest opening price. Return the ticker, the month (formatted as `Mon-YYYY`) and value of the highest opening price, along with the month and value of the lowest opening price.

**Schema columns:** `faang_stock_prices.date`, `faang_stock_prices.ticker`, `faang_stock_prices.open`, `faang_stock_prices.high`, `faang_stock_prices.low`, `faang_stock_prices.close`

**Output columns:** `ticker`, `highest_mth`, `highest_open`, `lowest_mth`, `lowest_open`

Order the result by `ticker` in ascending order.

### Examples

#### Example 1

**Input:**

**faang_stock_prices:**

| date | ticker | open | high | low | close |
|------------|--------|------:|------:|-----:|------:|
| 2023-01-31 | AAPL | 142.28 | 144.34 | 142.70 | 144.29 |
| 2023-03-31 | AAPL | 161.91 | 165.00 | 162.44 | 164.90 |
| 2023-05-31 | AAPL | 176.76 | 179.35 | 177.33 | 177.25 |
| 2023-01-31 | GOOG | 100.34 | 101.50 | 100.20 | 101.45 |
| 2023-03-31 | GOOG | 108.45 | 109.99 | 108.00 | 109.75 |
| 2023-05-31 | GOOG | 115.67 | 118.00 | 115.00 | 116.00 |

**Output:**

| ticker | highest_mth | highest_open | lowest_mth | lowest_open |
|--------|-------------|-------------:|------------|------------:|
| AAPL | May-2023 | 176.76 | Jan-2023 | 142.28 |
| GOOG | May-2023 | 115.67 | Jan-2023 | 100.34 |

**Explanation:** For each ticker, identify the maximum and minimum opening prices. Report the corresponding month formatted as `Mon-YYYY` along with the opening price.

### Constraints

- Format months as `Mon-YYYY` (for example, `Jan-2023`).
- Find the highest and lowest opening price for each ticker.
- Return one row per ticker.
- Order the result by `ticker`.

In [0]:
faang_stock_prices_data=[("2023-01-31","AAPL",142.28,144.34,142.70,144.29),("2023-03-31","AAPL",161.91,165.00,162.44,164.90),("2023-05-31","AAPL",176.76,179.35,177.33,177.25),("2023-01-31","GOOG",100.34,101.50,100.20,101.45),("2023-03-31","GOOG",108.45,109.99,108.00,109.75),("2023-05-31","GOOG",115.67,118.00,115.00,116.00)]

faang_stock_prices_df=spark.createDataFrame(faang_stock_prices_data,["date","ticker","open","high","low","close"])
display(faang_stock_prices_df)
w_high = Window.partitionBy("ticker").orderBy(desc("open"))
w_low = Window.partitionBy("ticker").orderBy("open")

highest = (
    faang_stock_prices_df
    .withColumn("rn", row_number().over(w_high))
    .filter(col("rn") == 1)
    .select(
        "ticker",
        date_format("date","MMM-yyyy").alias("highest_mth"),
        col("open").alias("highest_open")
    )
)

lowest = (
    faang_stock_prices_df
    .withColumn("rn", row_number().over(w_low))
    .filter(col("rn") == 1)
    .select(
        "ticker",
        date_format("date","MMM-yyyy").alias("lowest_mth"),
        col("open").alias("lowest_open")
    )
)

result = (
    highest.join(lowest, "ticker")
    .orderBy("ticker")
)

display(result)

date,ticker,open,high,low,close
2023-01-31,AAPL,142.28,144.34,142.7,144.29
2023-03-31,AAPL,161.91,165.0,162.44,164.9
2023-05-31,AAPL,176.76,179.35,177.33,177.25
2023-01-31,GOOG,100.34,101.5,100.2,101.45
2023-03-31,GOOG,108.45,109.99,108.0,109.75
2023-05-31,GOOG,115.67,118.0,115.0,116.0


ticker,highest_mth,highest_open,lowest_mth,lowest_open
AAPL,May-2023,176.76,Jan-2023,142.28
GOOG,May-2023,115.67,Jan-2023,100.34


%md
## Que4: DataFrame GroupBy with Aggregation

**Difficulty:** Medium

### Problem

A retail analytics team wants to identify the products generating the most revenue.

For each product, calculate the total revenue generated, the average selling price, and the total number of transactions. Revenue for each transaction is calculated as `price × quantity`. Round both `total_sales` and `avg_price` to 2 decimal places.

**Schema columns:** `sales.sale_id`, `sales.product_id`, `sales.price`, `sales.quantity`, `sales.sale_date`

**Output columns:** `product_id`, `total_sales`, `avg_price`, `transaction_count`

Order the result by `total_sales` in descending order.

### Examples

#### Example 1

**Input:**

**sales:**

| sale_id | product_id | price | quantity | sale_date |
|--------:|------------|------:|---------:|------------|
| 1 | P001 | 43.71 | 3 | 2024-01-01 |
| 2 | P002 | 95.56 | 7 | 2024-01-02 |
| 3 | P003 | 75.88 | 4 | 2024-01-03 |
| 5 | P002 | 24.04 | 3 | 2024-01-05 |
| 6 | P001 | 24.04 | 5 | 2024-01-06 |

**Output:**

| product_id | total_sales | avg_price | transaction_count |
|------------|------------:|----------:|------------------:|
| P002 | 741.04 | 59.80 | 2 |
| P003 | 303.52 | 75.88 | 1 |
| P001 | 251.33 | 33.88 | 2 |

**Explanation:** Revenue is calculated as `price × quantity` for each transaction. The revenues are aggregated by product, along with the average price and transaction count. The final result is sorted by total revenue in descending order.

### Constraints

- Return one row for each `product_id`.
- Calculate `total_sales` as the sum of `price × quantity`.
- Round both `total_sales` and `avg_price` to 2 decimal places.
- Count the number of transactions for each product.
- Order the result by `total_sales` in descending order.

In [0]:
sales_data=[(1,"P001",43.71,3,"2024-01-01"),(2,"P002",95.56,7,"2024-01-02"),(3,"P003",75.88,4,"2024-01-03"),(5,"P002",24.04,3,"2024-01-05"),(6,"P001",24.04,5,"2024-01-06")]
sales_df=spark.createDataFrame(sales_data,["sale_id","product_id","price","quantity","sale_date"])
display(sales_df)

output_df = (
sales_df.
    groupBy("product_id").agg(
        round(sum(col("price") * col("quantity")), 2).alias("total_sales"),
        round(avg("price"), 2).alias("avg_price"),
        count("sale_id").alias("transactions_count")
    )
    .orderBy(col("total_sales").desc())
)

display(output_df)


sale_id,product_id,price,quantity,sale_date
1,P001,43.71,3,2024-01-01
2,P002,95.56,7,2024-01-02
3,P003,75.88,4,2024-01-03
5,P002,24.04,3,2024-01-05
6,P001,24.04,5,2024-01-06


product_id,total_sales,avg_price,transactions_count
P002,741.04,59.8,2
P003,303.52,75.88,1
P001,251.33,33.88,2


%md
## Que5: Duplicate Payment Identification

**Difficulty:** Hard

### Problem

A payment system wants to detect duplicate credit card charges caused by accidental double-clicks or retry requests.

A transaction is considered a repeated payment if the immediately previous transaction with the same `merchant_id`, `credit_card_id`, and `amount` occurred within **10 minutes**. The first transaction in each group is always considered the original payment and should not be counted.

Return the total number of repeated payments.

**Schema columns:** `dpd_duplicate.transaction_id`, `dpd_duplicate.merchant_id`, `dpd_duplicate.credit_card_id`, `dpd_duplicate.amount`, `dpd_duplicate.transaction_timestamp`

**Output columns:** `payment_count`

### Examples

#### Example 1

**Input:**

**dpd_duplicate:**

| transaction_id | merchant_id | credit_card_id | amount | transaction_timestamp |
|---------------:|------------:|---------------:|-------:|-----------------------|
| 1 | 101 | 1 | 100 | 09/25/2022 12:00:00 |
| 2 | 101 | 1 | 100 | 09/25/2022 12:08:00 |
| 3 | 101 | 1 | 100 | 09/25/2022 12:28:00 |
| 4 | 102 | 2 | 300 | 09/25/2022 12:00:00 |
| 5 | 102 | 2 | 300 | 09/25/2022 12:05:00 |
| 6 | 102 | 2 | 400 | 09/25/2022 12:00:00 |

**Output:**

| payment_count |
|--------------:|
| 2 |

**Explanation:** Transaction 2 is a repeated payment because it occurred 8 minutes after transaction 1 with the same merchant, card, and amount. Transaction 5 is also a repeated payment because it occurred 5 minutes after transaction 4. Transaction 3 occurred 20 minutes after transaction 2, so it is not counted.

### Constraints

- Compare transactions having the same `merchant_id`, `credit_card_id`, and `amount`.
- Compare each transaction only with the immediately previous transaction in its group.
- A repeated payment must occur within 10 minutes of the previous transaction.
- The earliest transaction in each group is never counted.
- Return a single row containing `payment_count`.

In [0]:
dpd_duplicate_data=[(1,101,1,100,"09/25/2022 12:00:00"),(2,101,1,100,"09/25/2022 12:08:00"),(3,101,1,100,"09/25/2022 12:28:00"),(4,102,2,300,"09/25/2022 12:00:00"),(5,102,2,300,"09/25/2022 12:05:00"),(6,102,2,400,"09/25/2022 12:00:00")]

dpd_duplicate_df=spark.createDataFrame(dpd_duplicate_data,["transaction_id","merchant_id","credit_card_id","amount","transaction_timestamp"])

dpd_duplicate_df = dpd_duplicate_df.withColumn("transaction_timestamp", to_timestamp("transaction_timestamp", "MM/dd/yyyy HH:mm:ss"))

window = Window.partitionBy("merchant_id","credit_card_id", "amount").orderBy("transaction_timestamp")

time_diff_df = (
dpd_duplicate_df
    .withColumn("prev_timestamp", lag("transaction_timestamp").over(window))
    .withColumn("time_diff", timestamp_diff("minute", "prev_timestamp", "transaction_timestamp"))
    .filter((col("time_diff").isNotNull()) & (col("time_diff") <= 10))
)

output_df = time_diff_df.selectExpr("count(1) as payment_count")


display(output_df)

payment_count
2


%md
## Que6: Warehouse Inventory Optimization

**Difficulty:** Hard

### Problem

A warehouse has a total capacity of **500,000 square feet** and stores complete inventory sets.

Each inventory set contains one of every item belonging to the same `item_type`. First, maximize the number of complete `prime_eligible` sets that can fit in the warehouse. Then, use the remaining space to store as many complete `not_prime` sets as possible.

Return the total number of individual items stored for each item type.

**Schema columns:** `mis_inventory.item_id`, `mis_inventory.item_type`, `mis_inventory.item_category`, `mis_inventory.square_footage`

**Output columns:** `item_type`, `item_count`

Order the result by `item_type` in descending order.

### Examples

#### Example 1

**Input:**

**mis_inventory:**

| item_id | item_type | item_category | square_footage |
|--------:|------------|-------------------|---------------:|
| 4245 | not_prime | standing lamp | 26.4 |
| 1374 | prime_eligible | mini refrigerator | 68.0 |
| 2452 | prime_eligible | television | 85.0 |
| 3255 | not_prime | side table | 22.6 |
| 1672 | prime_eligible | laptop | 8.5 |

**Output:**

| item_type | item_count |
|------------|-----------:|
| prime_eligible | 9285 |
| not_prime | 6 |

**Explanation:** A complete prime-eligible set occupies 161.5 square feet, allowing 3,095 complete sets to fit first. The remaining space is then used to store three complete not-prime sets, resulting in six not-prime items.

### Constraints

- The warehouse capacity is **500,000 square feet**.
- Store as many complete `prime_eligible` sets as possible before storing `not_prime` sets.
- `item_count` equals the number of complete sets multiplied by the number of items in each set.
- Order the result by `item_type` in descending order.

In [0]:
mis_inventory_data=[(4245,"not_prime","standing lamp",26.4),(1374,"prime_eligible","mini refrigerator",68.0),(2452,"prime_eligible","television",85.0),(3255,"not_prime","side table",22.6),(1672,"prime_eligible","laptop",8.5)]
mis_inventory_df=spark.createDataFrame(mis_inventory_data,["item_id","item_type","item_category","square_footage"])

mis_inventory_df = mis_inventory_df.withColumn("square_footage", col("square_footage").cast("float"))

display(mis_inventory_df)

grouped_df = (
    mis_inventory_df
    .groupBy("item_type")
    .agg(
        sum("square_footage").alias("set_space"),
        count("*").alias("items_per_set")
    )
)

display(grouped_df)

prime = grouped_df.filter(col("item_type") == "prime_eligible").collect()[0]
non_prime = grouped_df.filter(col("item_type") == "not_prime").collect()[0]

prime_set_space = prime["set_space"]
prime_items_per_set = prime["items_per_set"]

non_prime_set_space = non_prime["set_space"]
non_prime_items_per_set = non_prime["items_per_set"]

capacity = 500000

prime_sets = int(capacity // prime_set_space)

remaining_space = capacity - (prime_sets * prime_set_space)

non_prime_sets = int(remaining_space // non_prime_set_space)

prime_item_count = prime_sets * prime_items_per_set
non_prime_item_count = non_prime_sets * non_prime_items_per_set

print("Prime item count:", prime_item_count)
print("Non-prime item count:", non_prime_item_count)


item_id,item_type,item_category,square_footage
4245,not_prime,standing lamp,26.4
1374,prime_eligible,mini refrigerator,68.0
2452,prime_eligible,television,85.0
3255,not_prime,side table,22.6
1672,prime_eligible,laptop,8.5


item_type,set_space,items_per_set
not_prime,49.0,2
prime_eligible,161.5,3


Prime item count: 9285
Non-prime item count: 6
